# CSE 579 Homework 1: Imitation Learning

In this assignment, you will implement:
1. **Behavior Cloning (BC)** with a Gaussian policy
2. **DAgger** (Dataset Aggregation)
3. **Autoregressive policy** for BC
4. **Diffusion policy** (extra credit)

You will train and evaluate on two environments: **Reacher** and **PointMaze**.

**Getting started:** Click **File > Save a copy in Drive** to create your own editable copy of this notebook. Do not edit the original.

## 1. Setup and Installation

In [ ]:
# Clone the homework repository
!git clone https://github.com/WEIRDLabUW/CSE-579-HW1.git
!cp -r CSE-579-HW1/* .
!cp -r CSE-579-HW1/.gitignore .

In [ ]:
# Install system dependencies (for MuJoCo rendering)
!apt-get install -y \
    libgl1-mesa-dev \
    libgl1-mesa-glx \
    libglew-dev \
    libosmesa6-dev \
    software-properties-common \
    patchelf

# Install PyTorch with CUDA support (separate to avoid index conflicts)
!pip install -q \
    torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 \
    --extra-index-url https://download.pytorch.org/whl/cu124

# Install mujoco first — latest has Python 3.12 prebuilt wheels.
# gymnasium-robotics 1.2.x pins mujoco<3.0, but those old versions lack
# cp312 wheels. We install mujoco separately, then gymnasium-robotics with
# --no-deps to avoid the downgrade, and manually install its other deps.
!pip install -q mujoco
!pip install -q "gymnasium-robotics>=1.2.0,<1.3.0" --no-deps
!pip install -q \
    gymnasium==0.29.1 \
    Jinja2 imageio PettingZoo \
    matplotlib \
    "numpy>=1.24,<2.0" \
    tqdm

import os
os.environ['LD_PRELOAD'] = ':/usr/lib/x86_64-linux-gnu/libGLEW.so'

# Restart the runtime so the new numpy version takes effect.
# After restart, skip this cell and continue from the next one.
os.kill(os.getpid(), 9)

**Important:** The cell above restarts the runtime to load the new numpy version. After it restarts, **do not re-run the install cell** — skip it and continue from the next cell.

In [ ]:
# Re-set env var (lost after runtime restart)
import os
os.environ['LD_PRELOAD'] = ':/usr/lib/x86_64-linux-gnu/libGLEW.so'

# Verify installation — expect output: (11,)
import gymnasium as gym
env = gym.make("Reacher-v4")
print(env.reset()[0].shape)
env.close()

## 2. Imports and Device Setup

In [ ]:
import torch
import torch.optim as optim
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import random
import math
import pickle
import matplotlib.pyplot as plt
from torch import distributions as pyd
from torch.distributions import Categorical
from torch.optim.lr_scheduler import LambdaLR
import collections
from typing import Any, Dict, Optional
from tqdm.auto import tqdm

from utils import (
    rollout, relabel_action, generate_paths, get_expert_data,
    PolicyGaussian, PolicyAutoRegressiveModel, mlp
)
from policy import (
    torchify_dict, ConditionalUnet1D, Policy, BaseDiffusionDataset,
    create_sample_indices, get_data_stats, normalize_data, unnormalize_data
)
import DiffusionPolicy as DiffusionPolicyModule
from DiffusionPolicy import DiffusionDataset
from evaluate import evaluate
import pytorch_utils as ptu
from reach_goal.envs.pointmaze_env import PointMazeEnv
from reach_goal.envs.pointmaze_expert import WaypointController

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

torch.manual_seed(0)
random.seed(0)
np.random.seed(0)

## 3. Behavior Cloning (BC)

Implement standard behavior cloning by maximizing log-likelihood of expert actions under the policy.

In [ ]:
"""
TODO: MODIFY TO FILL IN YOUR BC IMPLEMENTATION
"""
def simulate_policy_bc(env, policy, expert_data, num_epochs=500, episode_length=50,
                       batch_size=32):

    # Hint: Use standard pytorch supervised learning code to train the policy.
    optimizer = optim.Adam(list(policy.parameters()), lr=1e-4)
    all_obs = np.concatenate([d['observations'] for d in expert_data])
    all_actions = np.concatenate([d['actions'] for d in expert_data])
    num_samples = all_obs.shape[0]
    num_batches = num_samples // batch_size
    losses = []
    for epoch in range(num_epochs):
        ## TODO Students
        order = np.arange(num_samples)
        np.random.shuffle(order)
        running_loss = 0.0
        for i in range(num_batches):
            optimizer.zero_grad()
            #========== TODO: start ==========

            #========== TODO: end ==========
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        # if epoch % 10 == 0:
        print('[%d] loss: %.8f' %
            (epoch, running_loss / num_batches))
        losses.append(loss.item())

## 4. DAgger (Dataset Aggregation)

Implement DAgger: iteratively train a policy, roll it out, relabel the collected trajectories using the expert, and add them to the training set.

In [ ]:
"""
TODO: MODIFY TO FILL IN YOUR DAGGER IMPLEMENTATION
"""
def simulate_policy_dagger(env, policy, expert_paths, expert_policy=None, num_epochs=500, episode_length=50,
                            batch_size=32, num_dagger_iters=10, num_trajs_per_dagger=10):

    # Fill in your dagger implementation here.
    # Hint: Loop through num_dagger_iters iterations, at each iteration train a policy on the current dataset.
    # Then rollout the policy, use relabel_action to relabel the actions along the trajectory
    # with "expert_policy" and then add this to current dataset.
    # Repeat this so the dataset grows with states drawn from the policy, and relabeled actions using the expert.

    # Optimizer code
    optimizer = optim.Adam(list(policy.parameters()))
    losses = []
    returns = []

    trajs = expert_paths
    # Dagger iterations
    for dagger_itr in range(num_dagger_iters):
        all_obs = np.concatenate([d['observations'] for d in trajs])
        all_actions = np.concatenate([d['actions'] for d in trajs])
        num_samples = all_obs.shape[0]
        num_batches = num_samples // batch_size
        losses = []
        # Train the model with Adam
        for epoch in range(num_epochs):
            order = np.arange(num_samples)
            np.random.shuffle(order)
            running_loss = 0.0
            for i in range(num_batches):
                optimizer.zero_grad()
                #========== TODO: begin ==========

                #========== TODO: end ==========
                loss.backward()
                optimizer.step()

                # print statistics
                running_loss += loss.item()
            # if epoch % 10 == 0:
            print('[%d, %5d] loss: %.8f' %(epoch + 1, i + 1, running_loss/num_batches))
            losses.append(running_loss/num_batches)

        # Collecting more data for dagger
        trajs_recent = []
        for k in range(num_trajs_per_dagger):
            env.reset()
            #========== TODO: start ==========

            #========== TODO: end ==========

        trajs += trajs_recent
        mean_return = np.mean(np.array([traj['rewards'].sum() for traj in trajs_recent]))
        print("Average DAgger return is " + str(mean_return))
        returns.append(mean_return)

## 5. Autoregressive Policy

Implement the autoregressive policy that predicts each action dimension sequentially, conditioned on previous dimensions.

**Note:** In the standalone Python files, this class is defined in `utils.py`. Here we redefine it in the notebook for ease of editing.

In [ ]:
"""
TODO: MODIFY TO FILL IN YOUR AUTOREGRESSIVE POLICY IMPLEMENTATION
"""
class PolicyAutoRegressiveModel(nn.Module):
    def __init__(self, num_inputs, num_outputs, hidden_dim=65, hidden_depth=2, num_buckets=10, ac_low=-1, ac_high=1):
        super(PolicyAutoRegressiveModel, self).__init__()
        self.eps = 1e-8
        self.trunks = nn.ModuleList([mlp(num_inputs, hidden_dim, num_buckets, hidden_depth)] \
                        + [mlp(num_inputs + j + 1, hidden_dim, num_buckets, hidden_depth) for j in range(num_outputs - 1)])
        self.num_dims = num_outputs
        self.ac_low = torch.tensor(ac_low).to(device)
        self.ac_high = torch.tensor(ac_high).to(device)
        self.num_buckets = num_buckets
        self.bucket_size = torch.tensor((ac_high - ac_low) / num_buckets).to(device)

    def discretize(self, ac):
        bucket_idx = (ac - self.ac_low) // (self.bucket_size + self.eps)
        return torch.clip(bucket_idx, 0, self.num_buckets - 1)

    def undiscretize(self, bucket_idx, dimension):
        return_val = bucket_idx[:, None]*self.bucket_size + self.ac_low + self.bucket_size*0.5
        return return_val[:, dimension]

    def forward(self, state):
        vals = []
        log_probs = 0
        for j in range(self.num_dims):
            #========== TODO: start ==========

            #========== TODO: end ==========
        vals = torch.cat(vals, dim=-1)
        return vals, log_probs

    def log_prob(self, state, action):
        log_prob = 0.
        ac_discretized = self.discretize(action)
        for j in range(self.num_dims):
            #========== TODO: start ==========

            #========== TODO: end ==========

        return log_prob

## 6. Diffusion Policy (Extra Credit)

Implement the diffusion policy for the PointMaze environment.

In [ ]:
# =====================================================================
# Noise scheduler
# =====================================================================
class NoiseScheduler:
    """Discrete-time DDPM noise scheduler.

    Stores beta_t, alpha_t, and alpha_bar_t, and
    implements the forward (training) and reverse (sampling) DDPM steps.
    """

    def __init__(
        self,
        num_train_timesteps: int = 100,
        beta_schedule: str = "linear",
        beta_start: float = 1e-4,
        beta_end: float = 0.02,
        device: torch.device = torch.device("cuda"),
    ):
        self.num_train_timesteps = num_train_timesteps
        self.beta_schedule = beta_schedule
        self.device = device

        # ========== TODO 1: build the beta and alpha tables ==========
        # We need three tensors of length T = num_train_timesteps:
        #   betas      : the per-step forward variances beta_t
        #   alphas     : 1 - betas
        #   alphas_bar : cumulative product of alphas, i.e. prod_{s<=t} alpha_s
        # These will be used to build a "linear" schedule: betas linearly spaced from beta_start to beta_end.

        # ========== TODO 1: end ==========

        self.betas = betas.to(device=device)
        self.alphas = alphas.to(device=device)
        self.alphas_bar = alphas_bar.to(device=device)

    def add_noise(self, x_0, noise, timesteps):
        """Forward (noising) process closed form.

        Returns x_t given x_0, the noise epsilon, and a batch of integer
        timesteps in [0, T).

        x_0       : (B, T_h, A) clean samples
        noise     : (B, T_h, A) standard normal noise of the same shape
        timesteps : (B,) integer timesteps
        """
        # ========== TODO 2: forward closed-form sample ==========
        # The DDPM forward process q(x_t | x_0) is a closed-form Gaussian:
        #   x_t = sqrt(alpha_bar_t) * x_0 + sqrt(1 - alpha_bar_t) * noise
        # Index into self.alphas_bar with `timesteps` to get a (B,) tensor,
        # then add singleton dims so it broadcasts against x_0.

        # ========== TODO 2: end ==========
        return x_t

    @torch.no_grad()
    def step(self, noise_pred, t, x_t):
        """One reverse DDPM step.

        Returns x_{t-1} given x_t and the predicted noise eps_theta(x_t, t).
        Uses the direct epsilon-parameterized mean from Ho et al. 2020 eq. 11
        with sigma_t^2 = beta_t.
        """
        # ========== TODO 3: reverse DDPM step ==========
        # The mean of p_theta(x_{t-1} | x_t) is (Ho et al. eq. 11):
        #
        #   mu_theta = (1 / sqrt(alpha_t)) *
        #       (x_t  -  (beta_t / sqrt(1 - alpha_bar_t)) * eps_pred)
        #
        # The variance is simply sigma_t^2 = beta_t.
        #
        # For t > 0, sample  x_{t-1} = mu_theta + sqrt(beta_t) * z,  z ~ N(0,I)
        # For t == 0, return mu_theta directly (no noise).

        # ========== TODO 3: end ==========
        return x_prev


# =====================================================================
# Cosine LR with linear warmup.
# =====================================================================
def cosine_with_warmup(optimizer, num_warmup_steps, num_training_steps):
    def lr_lambda(current_step):
        if current_step < num_warmup_steps:
            return float(current_step) / float(max(1, num_warmup_steps))
        progress = float(current_step - num_warmup_steps) / float(
            max(1, num_training_steps - num_warmup_steps))
        return max(0.0, 0.5 * (1.0 + math.cos(math.pi * progress)))
    return LambdaLR(optimizer, lr_lambda)


# =====================================================================
# Diffusion policy
# =====================================================================
class DiffusionPolicy(Policy):
    def __init__(self, obs_size, obs_horizon, action_size, action_pred_horizon,
                 action_horizon, num_diffusion_iters=100, beta_schedule="linear",
                 device=torch.device('cuda')):
        self.device = device
        self.obs_horizon = obs_horizon
        self.action_size = action_size
        self.action_horizon = action_horizon
        self.action_pred_horizon = action_pred_horizon
        self.num_diffusion_iters = num_diffusion_iters

        self.net = ConditionalUnet1D(action_size, obs_size * obs_horizon).to(device)
        self.scheduler = NoiseScheduler(
            num_train_timesteps=num_diffusion_iters,
            beta_schedule=beta_schedule, device=device)

        self.obs_deque = collections.deque([], maxlen=self.obs_horizon)
        self.stats = None

    def set_stats(self, stats):
        self.stats = torchify_dict(stats, self.device)

    @torch.no_grad()
    def _process_obs(self, obs):
        obs = np.copy(obs)
        return {"state": torch.as_tensor(obs, dtype=torch.float32, device=self.device)}

    def reset(self):
        self.obs_deque.clear()

    def add_obs(self, obs):
        o = self._process_obs(obs)
        self.obs_deque.append(o)
        while len(self.obs_deque) < self.obs_horizon:
            self.obs_deque.append(o)

    def __call__(self, obs=None):
        return self.get_action(obs.squeeze() if obs is not None else None)

    @torch.no_grad()
    def get_action(self, obs=None):
        assert self.stats is not None
        if obs is not None:
            self.add_obs(obs)
        assert len(self.obs_deque) == self.obs_horizon
        states = torch.stack([x["state"] for x in self.obs_deque])

        # ========== TODO 5: sampling loop ==========
        # 1. Normalize the observation history with self.stats['state'] and
        #    flatten it to a (1, obs_horizon * obs_dim) conditioning tensor
        #    that the UNet expects via its `global_cond` argument.
        #
        # 2. Initialize a tensor of pure Gaussian noise with shape
        #    (1, action_pred_horizon, action_size). This is x_K, the most
        #    noised "action" we will progressively denoise.
        #
        # 3. Loop t = num_diffusion_iters - 1, ..., 0:
        #       a. Predict the noise: eps_pred = self.net(naction, t, obs_cond)
        #       b. Step:               naction = self.scheduler.step(
        #                                            eps_pred, t, naction)
        #
        # 4. Drop the batch dimension and unnormalize with
        #    self.stats['action'] to recover real-valued actions.
        #
        # 5. Slice out the action_horizon actions starting from index
        #    obs_horizon - 1 -- this is the "action chunk" we will execute
        #    before re-querying the policy.

        # ========== TODO 5: end ==========
        return action, {}

    def state_dict(self):
        return dict(net=self.net.state_dict(), stats=self.stats)

    def load_state_dict(self, state_dict):
        self.net.load_state_dict(state_dict["net"])
        self.set_stats(state_dict["stats"])


def train_diffusion_policy(policy, expert_data, num_epochs=500, batch_size=32):
    dataset = DiffusionDataset(expert_data, pred_horizon=policy.action_pred_horizon,
                               obs_horizon=policy.obs_horizon, action_horizon=policy.action_horizon)
    policy.set_stats(dataset.stats)

    data_loader = torch.utils.data.DataLoader(
        dataset, batch_size=batch_size, shuffle=True,
        num_workers=4, pin_memory=True, persistent_workers=True)

    optimizer = torch.optim.AdamW(policy.net.parameters(), lr=1e-4, weight_decay=1e-6)
    num_training_steps = len(data_loader) * num_epochs
    lr_scheduler = cosine_with_warmup(
        optimizer, num_warmup_steps=num_training_steps // 10,
        num_training_steps=num_training_steps)

    losses = []
    for epoch in tqdm(range(num_epochs)):
        running_loss = 0.0
        with tqdm(data_loader, leave=False) as tepoch:
            for batch in tepoch:
                naction = batch['action'].to(policy.device)
                nstate = batch['state'][:, :policy.obs_horizon].to(policy.device)
                B = nstate.shape[0]

                # ========== TODO 4: training step ==========
                # 1. Flatten the observation history into the global
                #    conditioning vector the UNet expects:
                #        obs_cond shape (B, obs_horizon * obs_dim)
                #
                # 2. Sample standard normal noise of the same shape as
                #    naction.
                #
                # 3. Sample one diffusion timestep per element of the batch:
                #        timesteps ~ Uniform{0, ..., T-1}, shape (B,)
                #
                # 4. Build the noisy actions x_t with the scheduler:
                #        noisy = policy.scheduler.add_noise(
                #            naction, noise, timesteps)
                #
                # 5. Predict the noise residual:
                #        noise_pred = policy.net(noisy, timesteps, obs_cond)
                #
                # 6. Compute MSE loss between predicted and true noise.

                # ========== TODO 4: end ==========

                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                lr_scheduler.step()

                loss_cpu = loss.item()
                running_loss += loss_cpu
                tepoch.set_postfix(loss=loss_cpu)
        epoch_loss = running_loss / len(data_loader)
        losses.append(epoch_loss)
        print(f"epoch {epoch} loss: {epoch_loss:.6f}")

    return policy

---
## 7. Training and Evaluation

Below we define helper functions and then run all the required training configurations.

In [ ]:
class Args:
    def __init__(self, env, train, policy, test=False, render=False):
        self.env = env       # 'reacher' or 'pointmaze'
        self.train = train   # 'behavior_cloning' or 'dagger' or 'diffusion'
        self.policy = policy # 'gaussian' or 'autoregressive' or 'diffusion'
        self.test = test
        self.render = render


def setup_and_run(args):
    """Load data, create env/policy, train, evaluate, and save weights."""
    # Get expert data
    if args.env == 'reacher':
        file_path = 'data/reacher_expert_data.pkl'
    elif args.env == 'pointmaze':
        file_path = 'data/pointmaze_expert_data.pkl'
    else:
        raise ValueError('Invalid environment')
    expert_data = get_expert_data(file_path)

    flattened_expert = {'observations': [], 'actions': []}
    for expert_path in expert_data:
        for k in flattened_expert.keys():
            flattened_expert[k].append(expert_path[k])
    for k in flattened_expert.keys():
        flattened_expert[k] = np.concatenate(flattened_expert[k])

    # Define environment
    if args.env == 'reacher':
        env = gym.make("Reacher-v4", render_mode='human' if args.render else None)
    elif args.env == 'pointmaze':
        env = PointMazeEnv(render_mode='human' if args.render else 'rgb_array')

    # Define policy
    hidden_dim = 128
    hidden_depth = 2
    obs_size = env.observation_space.shape[0]
    ac_size = env.action_space.shape[0]
    ac_margin = 0.1

    if args.policy == 'gaussian':
        policy = PolicyGaussian(num_inputs=obs_size, num_outputs=ac_size, hidden_dim=hidden_dim, hidden_depth=hidden_depth)
        policy.to(device)
    elif args.policy == 'autoregressive':
        num_buckets = 10
        policy = PolicyAutoRegressiveModel(num_inputs=obs_size, num_outputs=ac_size, hidden_dim=hidden_dim,
                                            hidden_depth=hidden_depth, num_buckets=num_buckets,
                                            ac_low=flattened_expert['actions'].min(axis=0) - ac_margin,
                                            ac_high=flattened_expert['actions'].max(axis=0) + ac_margin)
        policy.to(device)
    elif args.policy == 'diffusion':
        policy = DiffusionPolicy(obs_size=obs_size, obs_horizon=4, action_size=ac_size,
                                 action_horizon=8, action_pred_horizon=12,
                                 num_diffusion_iters=100, device=device)

    # Training hyperparameters
    if args.env == 'reacher':
        episode_length = 50
        num_epochs = 500
        batch_size = 32
    elif args.env == 'pointmaze':
        episode_length = 300
        num_epochs = 10
        batch_size = 128
        if args.train == 'diffusion':
            num_epochs = 1

    if not args.test:
        if args.train == 'behavior_cloning':
            simulate_policy_bc(env, policy, expert_data, num_epochs=num_epochs,
                               episode_length=episode_length, batch_size=batch_size)
        elif args.train == 'dagger':
            if args.env == 'reacher':
                expert_policy = torch.load('data/reacher_expert_policy.pkl', map_location=torch.device(device), weights_only=False)
                print("Expert policy loaded")
                expert_policy.to(device)
                ptu.set_gpu_mode(torch.cuda.is_available())
            elif args.env == 'pointmaze':
                expert_policy = WaypointController(env.maze)

            num_dagger_iters = 10
            num_epochs = int(num_epochs / num_dagger_iters)
            num_trajs_per_dagger = 10

            simulate_policy_dagger(env, policy, expert_data, expert_policy,
                                   num_epochs=num_epochs, episode_length=episode_length,
                                   batch_size=batch_size, num_dagger_iters=num_dagger_iters,
                                   num_trajs_per_dagger=num_trajs_per_dagger)
        elif args.train == 'diffusion' and args.env == 'pointmaze':
            policy = train_diffusion_policy(policy, get_expert_data(file_path),
                                           num_epochs=num_epochs, batch_size=batch_size)
        else:
            raise ValueError('Invalid training method')

        torch.save(policy.state_dict(), f'{args.policy}_{args.env}_{args.train}_final.pth')
    else:
        policy.load_state_dict(torch.load(f'{args.policy}_{args.env}_{args.train}_final.pth', weights_only=True))

    evaluate(env, policy, args.train, num_validation_runs=100, episode_length=episode_length,
             render=args.render, env_name=args.env)
    env.close()
    return policy

### 7a. Gaussian BC — Reacher

In [ ]:
args = Args('reacher', 'behavior_cloning', 'gaussian')
policy_gaussian_reacher_bc = setup_and_run(args)

### 7b. Gaussian DAgger — Reacher

In [ ]:
args = Args('reacher', 'dagger', 'gaussian')
policy_gaussian_reacher_dagger = setup_and_run(args)

### 7c. Gaussian BC — PointMaze

In [ ]:
args = Args('pointmaze', 'behavior_cloning', 'gaussian')
policy_gaussian_pointmaze_bc = setup_and_run(args)

### 7d. Gaussian DAgger — PointMaze

In [ ]:
args = Args('pointmaze', 'dagger', 'gaussian')
policy_gaussian_pointmaze_dagger = setup_and_run(args)

### 7e. Autoregressive BC — Reacher

In [ ]:
args = Args('reacher', 'behavior_cloning', 'autoregressive')
policy_autoregressive_reacher_bc = setup_and_run(args)

### 7f. Autoregressive BC — PointMaze

In [ ]:
args = Args('pointmaze', 'behavior_cloning', 'autoregressive')
policy_autoregressive_pointmaze_bc = setup_and_run(args)

### 7g. Diffusion Policy — PointMaze (Extra Credit)

In [ ]:
args = Args('pointmaze', 'diffusion', 'diffusion')
policy_diffusion_pointmaze = setup_and_run(args)

---
## 8. Submit to Autograder

After training all configurations, submit your code and weight files to Gradescope.

**If you worked in Colab:** Go to **File > Download > Download .py**, then upload that `.py` file along with your `.pth` weight files to the autograder. The autograder will automatically extract your code from the exported file.

**If you worked locally:** Submit `bc.py`, `dagger.py`, `utils.py`, `DiffusionPolicy.py`, and your `.pth` weight files.

Run the cell below to collect all weight files for download.

In [ ]:
import glob
import shutil

# List all saved weight files
weight_files = glob.glob('*_final.pth')
print("Found weight files:")
for f in sorted(weight_files):
    print(f"  {f}")

# Create a zip archive of weight files for easy download
submission_dir = 'submission'
os.makedirs(submission_dir, exist_ok=True)

for f in weight_files:
    shutil.copy(f, os.path.join(submission_dir, f))

shutil.make_archive('submission', 'zip', submission_dir)
print(f"\nCreated submission.zip with {len(weight_files)} weight files.")
print("\nTo submit to Gradescope:")
print("  1. Download this notebook as .py: File > Download > Download .py")
print("  2. Download submission.zip (or individual .pth files)")
print("  3. Upload both to Gradescope")

In [ ]:
# If running on Google Colab, trigger download of weight files
try:
    from google.colab import files
    files.download('submission.zip')
except ImportError:
    print("Not running on Colab. Please download submission.zip manually from the file browser.")